In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
class PPOAgent:
    def __init__(self, model_name, reward_model, tokenizer):
        self.policy_model = AutoModelForCausalLM.from_pretrained(model_name)
        self.reward_model = reward_model
        self.tokenizer = tokenizer
        self.ref_model = AutoModelForCausalLM.from_pretrained(model_name)  # Frozen reference
        
        # Freeze reference model
        for param in self.ref_model.parameters():
            param.requires_grad = False
    
    def generate(self, prompts, max_length=128, temperature=1.0):
        self.policy_model.eval()
        responses = []
        with torch.no_grad():
            for prompt in prompts:
                input_ids = self.tokenizer.encode(prompt, return_tensors='pt')
                
                outputs = self.policy_model.generate(
                    input_ids,
                    max_length=max_length,
                    temperature=temperature,
                    do_sample=True,
                    return_dict_in_generate=True,
                    output_scores=True
                )
                
                response = self.tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
                responses.append(response)
        
        return responses